#### Etape 3.2 : Analyse des correlations
- Calculer la matrice de correlation entre :
  - Consommations (electricite, gaz, eau)
  - Variables meteo (temperature, humidite, rayonnement, vent)
  - Caracteristiques batiments (surface, nb occupants, annee construction)
- Identifier les correlations significatives (>0.5 ou <-0.5)
- Analyser l'impact de la temperature sur la consommation de chauffage
- Etudier l'effet du rayonnement solaire sur la consommation electrique

**Livrables** :
- Notebook `07_analyse_correlations.ipynb`
- Matrice de correlation exportee `output/matrice_correlation.csv`
- Synthese des insights (format markdown)

In [42]:
import os


DATA_DIR = "../data"
DATA_DIR = os.path.join(DATA_DIR, "..", "data")
OUTPUT_DIR = os.path.join(DATA_DIR, "..", "output", "consommation_clean")
OUTPUT_METEO = os.path.join(DATA_DIR, "..", "output", "meteo_clean.csv")
DATA_TARIF = os.path.join(DATA_DIR, "..", "data", "tarifs_energie.csv")

OUTPUT_DIR_MATRIX = os.path.join(DATA_DIR, "..", "output", "matrice_correlation.csv")



In [43]:
import pandas as pd

df_conso = pd.read_parquet(OUTPUT_DIR)
df_meteo = pd.read_csv(OUTPUT_METEO)

print(df_conso.dtypes)
print(len(df_conso))
print()
print(df_meteo.dtypes)


batiment_id                str
unite                      str
conso_clean            float64
hour                     int32
year                     int32
month                    int32
nom                        str
type                       str
commune                    str
surface_m2               int32
annee_construction       int32
classe_energetique         str
nb_occupants_moyen       int32
date                  category
type_energie          category
dtype: object
7492584

commune                        str
timestamp                      str
temperature_c              float64
humidite_pct               float64
rayonnement_solaire_wm2    float64
vitesse_vent_kmh           float64
precipitation_mm           float64
date                           str
day_of_week                  int64
month                        int64
season                         str
hour                         int64
dtype: object


In [44]:
df_fusion_meteo = df_conso \
    .merge(
        df_meteo, on = ['date', 'hour', 'commune', 'month'],
        how = "left"
    )

print(df_fusion_meteo.dtypes)

batiment_id                     str
unite                           str
conso_clean                 float64
hour                          int32
year                          int32
month                         int32
nom                             str
type                            str
commune                         str
surface_m2                    int32
annee_construction            int32
classe_energetique              str
nb_occupants_moyen            int32
date                            str
type_energie               category
timestamp                       str
temperature_c               float64
humidite_pct                float64
rayonnement_solaire_wm2     float64
vitesse_vent_kmh            float64
precipitation_mm            float64
day_of_week                 float64
season                          str
dtype: object


In [45]:
#on flatmap les données

df_matrice = df_fusion_meteo \
    .groupby(
        [
            "batiment_id",
            "year",
            "annee_construction",
            "month",
            "commune",
            "surface_m2",
            "classe_energetique",
            "temperature_c",
            "humidite_pct",
            "rayonnement_solaire_wm2",
            "vitesse_vent_kmh",
            "precipitation_mm",
            "season",
            "day_of_week",
            "type_energie",
        ]
    )["conso_clean"] \
    .sum() \
    .unstack("type_energie") \
    .reset_index()

#on drop ce qui est pas numérique

df_matrice = df_matrice.drop(
    columns=["batiment_id", "year", "commune", "classe_energetique", "day_of_week", "season"]
)

print(df_matrice.dtypes)

type_energie
annee_construction           int32
month                        int32
surface_m2                   int32
temperature_c              float64
humidite_pct               float64
rayonnement_solaire_wm2    float64
vitesse_vent_kmh           float64
precipitation_mm           float64
eau                        float64
electricite                float64
gaz                        float64
dtype: object


In [46]:
matrice_corr = df_matrice.corr()
print(matrice_corr)

type_energie             annee_construction     month  surface_m2  \
type_energie                                                        
annee_construction                 1.000000 -0.000031   -0.012618   
month                             -0.000031  1.000000   -0.000257   
surface_m2                        -0.012618 -0.000257    1.000000   
temperature_c                      0.042401  0.075219    0.029913   
humidite_pct                      -0.000220 -0.005071    0.000092   
rayonnement_solaire_wm2            0.000050 -0.000668   -0.000374   
vitesse_vent_kmh                  -0.000083  0.000926   -0.000111   
precipitation_mm                   0.000263  0.000872   -0.000909   
eau                               -0.102789 -0.000389    0.633167   
electricite                       -0.146285 -0.005111    0.595706   
gaz                               -0.145615 -0.005401    0.597021   

type_energie             temperature_c  humidite_pct  rayonnement_solaire_wm2  \
type_energie         

In [47]:
# Colonnes pour la correlation
data_bat = ['annee_construction', 'surface_m2', 'eau', 'gaz', 'electricite']
data_meteo = ["month", "temperature_c", "humidite_pct", "rayonnement_solaire_wm2", "vitesse_vent_kmh", "precipitation_mm"]
available_data_bati = [p for p in data_bat if p in matrice_corr.columns]
available_meteo = [m for m in data_meteo if m in matrice_corr.columns]

corr_cols = available_data_bati + available_meteo

matrice_corr = matrice_corr.drop(
    index="type_energie",
    columns="type_energie",
    errors="ignore"
)

matrice_corr.to_csv(OUTPUT_DIR_MATRIX)


# Calculer la matrice de correlation
correlation_matrix = matrice_corr[corr_cols].corr().round(3)


print("Matrice de correlation:")
correlation_matrix



Matrice de correlation:


type_energie,annee_construction,surface_m2,eau,gaz,electricite,month,temperature_c,humidite_pct,rayonnement_solaire_wm2,vitesse_vent_kmh,precipitation_mm
type_energie,,,,,,,,,,,
annee_construction,1.000,-0.360,-0.481,-0.501,-0.501,-0.059,0.069,-0.061,-0.178,-0.060,-0.060
surface_m2,-0.360,1.000,0.885,0.856,0.855,-0.250,-0.279,-0.227,-0.051,-0.232,-0.233
eau,-0.481,0.885,1.000,0.979,0.979,-0.262,-0.349,-0.234,0.147,-0.241,-0.240
gaz,-0.501,0.856,0.979,1.000,0.995,-0.255,-0.450,-0.216,0.208,-0.223,-0.222
electricite,-0.501,0.855,0.979,0.995,1.000,-0.254,-0.450,-0.217,0.209,-0.223,-0.222
month,-0.059,-0.250,-0.262,-0.255,-0.254,1.000,0.069,-0.117,-0.174,-0.104,-0.104
temperature_c,0.069,-0.279,-0.349,-0.450,-0.450,0.069,1.000,-0.088,-0.190,-0.089,-0.081
humidite_pct,-0.061,-0.227,-0.234,-0.216,-0.217,-0.117,-0.088,1.000,-0.161,-0.100,-0.103
rayonnement_solaire_wm2,-0.178,-0.051,0.147,0.208,0.209,-0.174,-0.190,-0.161,1.000,-0.159,-0.164


In [48]:
corr_significatives = correlation_matrix \
    .where((correlation_matrix > 0.4) | (correlation_matrix < -0.4)) 

print("Corrélations ok (|corr| > 0.5) :")
print(corr_significatives)

Corrélations ok (|corr| > 0.5) :
type_energie             annee_construction  surface_m2    eau    gaz  \
type_energie                                                            
annee_construction                    1.000         NaN -0.481 -0.501   
surface_m2                              NaN       1.000  0.885  0.856   
eau                                  -0.481       0.885  1.000  0.979   
gaz                                  -0.501       0.856  0.979  1.000   
electricite                          -0.501       0.855  0.979  0.995   
month                                   NaN         NaN    NaN    NaN   
temperature_c                           NaN         NaN    NaN -0.450   
humidite_pct                            NaN         NaN    NaN    NaN   
rayonnement_solaire_wm2                 NaN         NaN    NaN    NaN   
vitesse_vent_kmh                        NaN         NaN    NaN    NaN   
precipitation_mm                        NaN         NaN    NaN    NaN   

type_energie     

#### On peut voir la corrélation négative entre la consommation d'énergie et la température (-0.45). La consommation augmente lorsque la température baisse.